## 📖 Libro: §1.3 + §1.4 del Capítulo 1 — Métrica de Fisher como tensor Riemanniano

**Enunciado (verbatim del libro):** Implemente un cálculo numérico de la métrica de Fisher para $\mathrm{Bern}(p)$ usando `np.random.default_rng`. Verifique que $I(p) = 1/(p(1-p))$ se cumple empíricamente con $n=10^4$ muestras, y que el gráfico de la métrica tiene divergencia en los bordes (p → 0 y p → 1).

**@ Pregunta a tu LLM:** «¿Por qué la métrica de Fisher para Bernoulli diverge en los extremos — qué significa geométricamente eso sobre la curvatura del espacio de Bernoulli?»

In [ ]:
# =====================================================================
# Celdas 1: imports + seed determinístico (cross-platform via SHA-256)
# =====================================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # notebooks/ -> raíz

import numpy as np
import matplotlib.pyplot as plt

from utils import setup_seed, plot_fisher_bernoulli, load_or_generate

SEED = setup_seed("cap1_fisher_tensor")
rng = np.random.default_rng(SEED)
print(f"Seed determinístico: {SEED}")

In [ ]:
# =====================================================================
# Celda 2: Verificación cuantitativa de I(p) sobre Bernoulli
# =====================================================================
# Para cada p en una grilla fina, muestreamos n Bernoulli(p) y estimamos
# I_emp(p) = var(score_empírico). Comparamos con I_teo(p) = 1/(p(1-p)).
def empirical_fisher_bernoulli(p: float, n: int = 10_000) -> float:
    samples = rng.binomial(1, p, size=n)
    # score = x/p - (1-x)/(1-p)
    scores = samples / p - (1.0 - samples) / (1.0 - p)
    return float(np.var(scores))

ps = [0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95]
print(f"{'p':>6s}  {'I_teo':>12s}  {'I_emp':>12s}  {'rel_err':>10s}")
print("-" * 48)
for p in ps:
    I_teo = 1.0 / (p * (1.0 - p))
    I_emp = empirical_fisher_bernoulli(p, n=10_000)
    rel_err = abs(I_emp - I_teo) / I_teo
    print(f"{p:>6.2f}  {I_teo:>12.4f}  {I_emp:>12.4f}  {rel_err:>10.2%}")

# Esperado (n grande): rel_err < 10% en todos los puntos. En p=0.5 la convergencia es más rápida.

In [ ]:
# =====================================================================
# Celda 3: Plot de la métrica I(p) (canvas oscura + curva Accent)
# =====================================================================
fig = plot_fisher_bernoulli(p_max=0.95, figsize=(8.5, 4.0),
                            title=r"Métrica de Fisher para Bernoulli: $I(p)=\dfrac{1}{p(1-p)}$")
fig.patch.set_facecolor("#FAFAFA")
plt.show()

In [ ]:
# =====================================================================
# Celda 4: Verificación de transformación tensorial bajo reparametrización
# logit η = log(p/(1-p)) ⇒ g^η = (dp/dη)² · g^p = p²(1-p)² · 1/(p(1-p)) = p(1-p)
# =====================================================================
print("Verificación de invariancia tensorial bajo logit:\n")
print(f"{'p':>6s}  {'η':>10s}  {'g_p(p)':>10s}  {'g_η(η)':>10s}  {'|diff|':>10s}")
print("-" * 52)
for p in ps:
    eta = np.log(p / (1 - p))
    g_p = 1.0 / (p * (1.0 - p))
    g_eta = p * (1.0 - p)  # = E[(∂ℓ/∂η)²] usando dp/dη = p(1-p)
    diff = abs(g_p * (p * (1 - p))**2 - g_eta)  # Should be ~0
    print(f"{p:>6.2f}  {eta:>10.4f}  {g_p:>10.4f}  {g_eta:>10.4f}  {diff:>10.2e}")

# Esperado: g_p(p) · (dp/dη)² == g_η(η), es decir, la métrica se
# transforma como un tensor covariante de orden 2. La columna |diff|
# debe ser ≈ 0 en todos los puntos (error numérico de doble precisión).

## ✅ `@ Verifica con:`

Debería cumplirse:
1. **Columna `rel_err`**: < 10% para todos los $p \in \{0.1, 0.3, 0.5, 0.7, 0.9\}$; en $p=0.5$ debe ser < 1%.
2. **Gráfico de $I(p)$**: divergencia en $p \to 0$ y $p \to 1$, mínimo en $p=0.5$ con valor $4.0$. (Coincide con `Respuestas.tex` §Cap.~1, ítem 7: $1/(0{.}5 \cdot 0{.}5)=4$.)
3. **Columna `|diff|`**: $\le 10^{-10}$ en todos los $p$, verificando la invariancia tensorial de la métrica bajo el cambio de coordenadas logit.

Conexión con el libro:
- §1.3: la métrica de Fisher como tensor Riemanniano.
- §1.4: invariancia bajo reparametrización (cambio logit).
- `respuestas.tex` §Cap.~1, ítems 16 y 17 verifican numéricamente la transformación tensorial.